# Install Kaggle API

In [5]:
%pip install kagglehub

Note: you may need to restart the kernel to use updated packages.


# Download Kaggle dataset

In [42]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("wordsforthewise/lending-club")

print("Path to dataset files:", path)

100%|██████████| 1.26G/1.26G [01:59<00:00, 11.3MB/s]

Extracting files...


Path to dataset files: /Users/meow/.cache/kagglehub/datasets/wordsforthewise/lending-club/versions/3


# Requirements

In [43]:
import re
import os

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# "magic" command to make plots show up in the notebook
%matplotlib inline 

# Data analysis

In [44]:
folders = os.listdir(path)
# Skip .xslx file, if we are ever able to upload it...
# Currently can't upload due to conflicts with other versions of the dataset on Kaggle.
folders = [f for f in folders if 'xlsx' not in f]
folders

['rejected_2007_to_2018q4.csv',
 'accepted_2007_to_2018q4.csv',
 'rejected_2007_to_2018Q4.csv.gz',
 'accepted_2007_to_2018Q4.csv.gz']

In [ ]:
os.listdir(path + "/" + folders[1])

['accepted_2007_to_2018Q4.csv']

In [50]:
acc_folder = path + "/" + [f for f in folders if 'accepted' in f][0]
accepted_fn = acc_folder + '/' + os.listdir(acc_folder)[0]

rej_folder = path + "/" + [f for f in folders if 'rejected' in f][0]
rejected_fn = rej_folder + '/' + os.listdir(rej_folder)[0]

accepted_fn

'/Users/meow/.cache/kagglehub/datasets/wordsforthewise/lending-club/versions/3/accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv'

#### check if the actual file is still there

In [ ]:
if os.path.isfile(accepted_fn) and os.path.isfile(rejected_fn):
    print('both paths still point to the actual file; all is good')
else:
    print('Kaggle changed how they handle compressed files again...you need to locate the files')

both paths still point to the actual file; all is good


In [52]:
# Takes a while to read, because these files are large...give it a minute or so
acc_df = pd.read_csv(accepted_fn)

# this is a dataset with rejected loans from lendingclub
rej_df = pd.read_csv(rejected_fn)

/var/folders/jp/4_3j1ct56glgcp_62fr2zb3c0000gn/T/ipykernel_46603/3491062877.py:2: DtypeWarning: Columns (0,19,49,59,118,129,130,131,134,135,136,139,145,146,147) have mixed types. Specify dtype option on import or set low_memory=False.
  acc_df = pd.read_csv(accepted_fn)


In [ ]:
acc_df.shape 

(2260701, 151)

* 2260701 rows, 151 columns in accepted file

In [54]:
rej_df.shape

(27648741, 9)

* 27648741 rows, 9 columns in accepted file

### FICO Score
- FICO stands for Fair Isaac Corporation, the company that created one of the most widely used credit scoring systems in the United States.

- A FICO score is a numerical measure of a person's creditworthiness — basically, how likely they are to repay a loan.

In [ ]:
# fico score in accepted loans
[col for col in acc_df.columns if 'fico' in col.lower()]

['fico_range_low',
 'fico_range_high',
 'last_fico_range_high',
 'last_fico_range_low',
 'sec_app_fico_range_low',
 'sec_app_fico_range_high']

In [ ]:
# fico score in rejected loans
[col for col in rej_df.columns if 'fico' in col.lower()]

[]

### Rejected information

In [57]:
rej_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27648741 entries, 0 to 27648740
Data columns (total 9 columns):
 #   Column                Dtype  
---  ------                -----  
 0   Amount Requested      float64
 1   Application Date      object 
 2   Loan Title            object 
 3   Risk_Score            float64
 4   Debt-To-Income Ratio  object 
 5   Zip Code              object 
 6   State                 object 
 7   Employment Length     object 
 8   Policy Code           float64
dtypes: float64(3), object(6)
memory usage: 1.9+ GB


### Accepted information

In [ ]:
acc_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Columns: 151 entries, id to settlement_term
dtypes: float64(113), object(38)
memory usage: 2.5+ GB


In [78]:
pd.options.display.max_rows

1000

In [79]:
acc_df.columns

Index(['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv',
       'term', 'int_rate', 'installment', 'grade', 'sub_grade',
       ...
       'hardship_payoff_balance_amount', 'hardship_last_payment_amount',
       'disbursement_method', 'debt_settlement_flag',
       'debt_settlement_flag_date', 'settlement_status', 'settlement_date',
       'settlement_amount', 'settlement_percentage', 'settlement_term'],
      dtype='object', length=151)

In [80]:
# update max number of rows
pd.options.display.max_rows = 1000

In [76]:
acc_df.head().T

,0,1,2,3,4
id,68407277,68355089,68341763,66310712,68476807
member_id,NaN,NaN,NaN,NaN,NaN
loan_amnt,3600.0,24700.0,20000.0,35000.0,10400.0
funded_amnt,3600.0,24700.0,20000.0,35000.0,10400.0
funded_amnt_inv,3600.0,24700.0,20000.0,35000.0,10400.0
term,36 months,36 months,60 months,60 months,60 months
int_rate,13.99,11.99,10.78,14.85,22.45
installment,123.03,820.28,432.66,829.9,289.91
grade,C,C,B,C,F
sub_grade,C4,C1,B4,C5,F1


In [85]:
# .info() tells us the datatype(int64, `object` is a string)
# and will also tell us the number of non-null (not missing) data points for each column
# because this dataframe is so large, we have to force it to show the datatypes and non-null numbers with the arguments
acc_df.info(verbose = True, max_cols=None)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Data columns (total 151 columns):
 #    Column                                      Dtype  
---   ------                                      -----  
 0    id                                          object 
 1    member_id                                   float64
 2    loan_amnt                                   float64
 3    funded_amnt                                 float64
 4    funded_amnt_inv                             float64
 5    term                                        object 
 6    int_rate                                    float64
 7    installment                                 float64
 8    grade                                       object 
 9    sub_grade                                   object 
 10   emp_title                                   object 
 11   emp_length                                  object 
 12   home_ownership                              object 
 13   annual_inc

In [ ]:
# Drop columns that leak future information/ irrelevant

cols_to_drop = [
    # ID & Metadata - Unique identifiers and verbose descriptions that don't generalise for modeling
    'id', 'member_id', 'url', 'desc', 'zip_code', 

    # Outstanding Balance - Remaining balance is a result of the loan, not an input feature
    'out_prncp', 'out_prncp_inv', 

    # Payment History / Loan Servicing - Values known after the loan is underway
    'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'total_rec_hist',
    'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 
    
    # Collections/Recovery
    'recoveries', 'collection_recovery_fee', 
    'hardship_flag', 'disbursement_method', 'debt_settlement_flag', 'settlement_status', 'settlement_date', 'settlement_amount', 'settlement_percentage', 'settlement_term'

    # Current metrics performance
    'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'num_accts_ever_120_pd', 'tot_coll_amt', 'tot_cur_bal'
]


print("Attempting to load the entire dataset...")

Attempting to load the entire dataset...


In [29]:
df_header = pd.read_csv(
    f"{path}/Loan_status_2007-2020Q3.gzip", 
    nrows=0  # Read 0 rows (header only)
)

# Convert the index (column names) to a list
all_columns = df_header.columns.tolist()

print(f"Total columns found: {len(all_columns)}")
# print(all_columns) # Optional: print the list of all column names

Total columns found: 142


In [31]:
# Convert lists to sets for the difference operation
set_all_columns = set(all_columns)
set_cols_to_drop = set(cols_to_drop)

# Find columns in ALL that are NOT in DROP
set_cols_to_retain = set_all_columns - set_cols_to_drop

# Convert the result back to a list, which will be your final list for usecols
final_cols_to_retain = list(set_cols_to_retain)

print(f"Columns to be retained: {len(final_cols_to_retain)}")
# print(final_cols_to_retain) # Optional: print the final list

Columns to be retained: 122


In [32]:
# read data
df = pd.read_csv(f"{path}/Loan_status_2007-2020Q3.gzip", usecols=final_cols_to_retain)

print("\n--- Final DataFrame Loaded ---")
print(f"DataFrame Shape: {df.shape}")
print(df.head())


/var/folders/jp/4_3j1ct56glgcp_62fr2zb3c0000gn/T/ipykernel_46603/1845665299.py:2: DtypeWarning: Columns (58,117,127,128,129,132,133,134,137) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{path}/Loan_status_2007-2020Q3.gzip", usecols=final_cols_to_retain)



--- Final DataFrame Loaded ---
DataFrame Shape: (2925493, 122)
   Unnamed: 0  loan_amnt  funded_amnt  funded_amnt_inv        term int_rate  \
0           0     5000.0       5000.0           4975.0   36 months   10.65%   
1           1     2500.0       2500.0           2500.0   60 months   15.27%   
2           2     2400.0       2400.0           2400.0   36 months   15.96%   
3           3    10000.0      10000.0          10000.0   36 months   13.49%   
4           4     3000.0       3000.0           3000.0   60 months   12.69%   

   installment grade sub_grade                 emp_title  ... hardship_amount  \
0       162.87     B        B2                       NaN  ...             NaN   
1        59.83     C        C4                     Ryder  ...             NaN   
2        84.33     C        C5                       NaN  ...             NaN   
3       339.31     C        C1       AIR RESOURCES BOARD  ...             NaN   
4        67.79     B        B5  University Medical Group

In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2925493 entries, 0 to 2925492
Columns: 122 entries, Unnamed: 0 to hardship_last_payment_amount
dtypes: float64(93), int64(1), object(28)
memory usage: 2.7+ GB


In [41]:
missing_pct = df.isnull().sum() / len(df) * 100
high_missing_cols = missing_pct[missing_pct > 50].sort_values(ascending=False)
print(high_missing_cols)
print(f"\nTotal number of high-missing columns: {len(high_missing_cols)}")

hardship_loan_status                          95.097886
hardship_reason                               95.090332
hardship_status                               95.090229
hardship_dpd                                  95.090161
payment_plan_start_date                       95.090127
hardship_end_date                             95.090127
deferral_term                                 95.090127
hardship_start_date                           95.090127
hardship_type                                 95.090127
hardship_length                               95.090127
orig_projected_additional_accrued_interest    93.873169
hardship_amount                               93.776228
hardship_payoff_balance_amount                93.776228
hardship_last_payment_amount                  93.776228
sec_app_revol_util                            93.348574
verification_status_joint                     93.341738
revol_bal_joint                               93.237960
sec_app_collections_12_mths_ex_med            93

In [35]:
print(df['loan_status'].value_counts())

loan_status
Fully Paid                                             1497783
Current                                                1031016
Charged Off                                             362548
Late (31-120 days)                                       16154
In Grace Period                                          10028
Late (16-30 days)                                         2719
Issued                                                    2062
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                    433
Name: count, dtype: int64
